# Controlled experiments for the teaser judge (task YL-004B, lab)

One experiment = the same set of teasers with known errors, **one change at a time** (the rubric file or the model), repeated several times.
Every run is logged in MLflow (in a folder you download at the end; a Google Drive is optional) and in a simple table, so you can compare versions.

Rule: change **one** thing between experiments. Three repeats per version, because the same input can give slightly different answers.

Not tested yet: the MLflow part (the package is not available in the cloud sandbox). If a cell fails, send me the error text.

**Since 6 Nov:** the answers are scored against the standard v2 (every sentence of every teaser marked by hand), and the statistics are given **per teaser**. The old count "found N of 42" is discarded. Steps 5 to 7d were run on 5 Nov and their results are re-scored from the saved archive; in a new session start at step 7e. Results go to the folder OUT (on Drive if `USE_DRIVE = True`); MLflow gets the same v2 numbers per teaser.

### Step 1. Code, packages and the MLflow folder
By default the runs are kept in a temporary folder; the last step downloads them. No Google Drive access is needed.

In [ ]:
import os
# MLflow folder (STORE) and the results (OUT: experiments_v2.csv and experiment_runs/). By default temporary folders in Colab.
# Set USE_DRIVE = True to keep both on a Drive: Colab then asks for FULL access to the Drive of the
# account that runs this Colab (a Drive of another account cannot be mounted).
USE_DRIVE = False
STORE = "/content/mlruns"
OUT = "/content/claim-check/results"
if USE_DRIVE:
    try:
        from google.colab import drive
        drive.mount("/content/drive")
        STORE = "/content/drive/MyDrive/teaser_judge_mlruns"
        OUT = "/content/drive/MyDrive/teaser_judge_results"
    except Exception as error:
        print("Google Drive not connected, using the temporary folder:", error)
os.makedirs(STORE, exist_ok=True)
os.makedirs(OUT, exist_ok=True)
print("MLflow folder:", STORE)
print("Results folder:", OUT)

REPO_URL = "https://github.com/Yul04ek/ai-content-claim-check.git"
BRANCH = "lab-step3"
FOLDER = "/content/claim-check"

if not os.path.exists(FOLDER):
    try:
        from google.colab import userdata
        token = userdata.get("GITHUB_TOKEN")
        clone_url = REPO_URL.replace("https://", f"https://{token}@")
    except Exception:
        clone_url = REPO_URL
    !git clone -q -b {BRANCH} {clone_url} {FOLDER}
    !git -C {FOLDER} remote set-url origin {REPO_URL}
else:
    !git -C {FOLDER} fetch -q origin {BRANCH}
    !git -C {FOLDER} checkout -q {BRANCH}
    !git -C {FOLDER} pull -q origin {BRANCH}
%cd {FOLDER}
!git log --oneline -1
!pip install -q -r requirements.txt mlflow pandas
import mlflow; print("mlflow", mlflow.__version__)

### Step 2. Connect the model
Should print `ready`.

In [ ]:
import os
from google.colab import userdata

# Read the key from Colab Secrets
try:
    os.environ["API_KEY"] = userdata.get("GEMINI_API_KEY")
except Exception:
    raise RuntimeError(
        "Secret GEMINI_API_KEY not found. Add it with the key icon in the left sidebar, "
        "switch on 'Notebook access', and run this cell again."
    )

os.environ["BASE_URL"] = "https://generativelanguage.googleapis.com/v1beta/openai/"  # Gemini through the OpenAI-compatible API
os.environ["MODEL_NAME"] = "gemini-3.5-flash-lite"   # the model that worked in the research agent; change if needed

# Send one short message to check that the key and the model work. Waits and tries again when the model is busy.
import time
from openai import OpenAI, APIStatusError

client = OpenAI(base_url=os.environ["BASE_URL"], api_key=os.environ["API_KEY"]).with_options(timeout=30, max_retries=0)
for attempt in range(5):
    try:
        reply = client.chat.completions.create(
            model=os.environ["MODEL_NAME"],
            messages=[{"role": "user", "content": "Reply with the single word: ready"}],
        )
        print(reply.choices[0].message.content)
        break
    except APIStatusError as error:
        # 503 = the model is overloaded, 429 = per-minute limit. Both are temporary (404 would mean a wrong model name).
        if error.status_code in (429, 500, 502, 503, 504) and attempt < 4:
            wait = 20 * (attempt + 1)
            print(f"Model answered {error.status_code}, waiting {wait} s ...")
            time.sleep(wait)
        else:
            raise

### Step 3. Upload the lab set
One zip file `lab_set_for_colab.zip` (four teasers, their evidence packs, the standard v2 `gold_v2.json` and the old list of known errors). It stays in Colab only, never in the repository.

In [ ]:
%cd /content/claim-check
import os, glob, zipfile
from google.colab import files

uploaded = files.upload()   # choose lab_set_for_colab.zip
os.makedirs("/content/lab_set", exist_ok=True)
for name in uploaded:
    if name.endswith(".zip"):
        zipfile.ZipFile(name).extractall("/content/lab_set")
LAB = glob.glob("/content/lab_set/**/lab_set.json", recursive=True)[0]
print("Lab set:", LAB)
GOLD = os.path.join(os.path.dirname(LAB), "gold_v2.json")
print("Standard v2 found:", os.path.exists(GOLD))
assert os.path.exists(GOLD), "gold_v2.json is missing: upload lab_set_for_colab.zip (not lab_set_v1.zip)"

### Step 4. Free check of the setup (no model call)
Prints the statistics per teaser with the code rules alone: `read 0` (no judge) and the number of code findings. Nothing is saved (no table, no run folders, no MLflow).

In [ ]:
%cd /content/claim-check
!python tests/test_teaser_check.py | tail -1
!python -m teaser_check.experiment "{LAB}" --dry-run --repeats 1

### Step 5. Baseline: the current rubric, 3 repeats
This is the starting point. It calls the model about 12 times (4 teasers x 3 repeats), a few cents on the paid price, free on the free tier.
Each repeat is one run in MLflow; the parent run `teaser-judge@<model>` has the mean, minimum and maximum recall and a hit rate per error (1.0 = found every time).

(Optional now: the baseline t1 of 5 Nov is in the old archive, see the optional cell in step 6. You can skip steps 5 to 7d in a new session.)

In [ ]:
%cd /content/claim-check
!python -m teaser_check.experiment "{LAB}" --rubric skills/teaser-judge.md --repeats 3 --store "{STORE}" --out-dir "{OUT}" --experiment teaser-judge

### Step 6. Compare (per teaser)
`score_v2` counts every saved run folder against the standard v2: how many statements the judge read, how many errors it found, false flags, right verdicts, code findings. Per teaser first.
To add the runs of 5 Nov: run the optional cell below and choose `teaser_experiments_8.zip` (the last archive you downloaded).

In [ ]:
%cd /content/claim-check
!python -m teaser_check.score_v2 "{GOLD}" "{OUT}/experiment_runs" --after 00-00-00

In [ ]:
# OPTIONAL: bring the runs of 5 Nov into this session (choose teaser_experiments_8.zip), then run the cell above again
%cd /content/claim-check
import zipfile
from google.colab import files
for name in files.upload():
    if name.endswith(".zip"):
        zipfile.ZipFile(name).extractall("/content/old_runs")
!mkdir -p "{OUT}/experiment_runs" && cp -rn /content/old_runs/results/experiment_runs/* "{OUT}/experiment_runs/"

### Step 7. A new version of the rubric (t3: t2 plus a fix for hypothesis_unlabelled)
Run only the new rubric: the baseline (t1) is already in the table. Compare rows with the same model.
The line `rubrics ['t3']` in the output shows that the new file was used.

In [ ]:
%cd /content/claim-check
!git pull -q origin lab-step3
!git log --oneline -1
import os
assert os.path.exists("skills/experiments/t3.md"), "t3.md is missing: git pull did not bring it"
!python -m teaser_check.experiment "{LAB}" --rubric skills/experiments/t3.md --repeats 3 --store "{STORE}" --out-dir "{OUT}"

### Step 7b. Part-by-part judge (rubric s1)
The teaser is split into its parts (snapshot, hypotheses, AURA help) and each part is judged with its own rule. The rules are those of t3, sorted by part, so the only change is the structure. The line `rubrics ['s1']` shows that the new file was used.

In [ ]:
%cd /content/claim-check
!git pull -q origin lab-step3
!git log --oneline -1
import os
assert os.path.exists("skills/experiments/s1.md"), "s1.md is missing: git pull did not bring it"
!python -m teaser_check.experiment "{LAB}" --rubric skills/experiments/s1.md --repeats 3 --store "{STORE}" --out-dir "{OUT}"

### Step 7c. Rubric t4 (hypotheses judged as a possibility)
Only the hypotheses block differs from t3: the company side is checked against the pack, the AURA side against the approved offers, and the wording must be a possibility. Look for `rubrics ['t4']`. Open `report_D.html` etc. in the run folder for the readable verdict.

In [ ]:
%cd /content/claim-check
!git pull -q origin lab-step3
import os
assert os.path.exists("skills/experiments/t4.md"), "t4.md is missing: git pull did not bring it"
!python -m teaser_check.experiment "{LAB}" --rubric skills/experiments/t4.md --repeats 3 --store "{STORE}" --out-dir "{OUT}"

### Step 7d. Rubric t4b (t4 + no sentence under a hypothesis is skipped)
In t4 the judge checked 10-20 statements fewer than in t3. t4b states that every sentence under a hypothesis must get an entry. Compare `judged` and the found/other numbers with t3 and t4. Look for `rubrics ['t4b']`.

In [ ]:
%cd /content/claim-check
!git pull -q origin lab-step3
import os
assert os.path.exists("skills/experiments/t4b.md"), "t4b.md is missing: git pull did not bring it"
!python -m teaser_check.experiment "{LAB}" --rubric skills/experiments/t4b.md --repeats 3 --store "{STORE}" --out-dir "{OUT}"

### Step 7e. Rubric t3n (t3 without the finding hypothesis_unlabelled), model gemini-3.1-flash-lite
One change against t3: the judge may no longer choose `hypothesis_unlabelled` (its wording rule is gone too). Everything else is as in t3. Look for `rubrics ['t3n']`.

In [ ]:
%cd /content/claim-check
!git pull -q origin lab-step3
import os
assert os.path.exists("skills/experiments/t3n.md"), "t3n.md is missing: git pull did not bring it"
!python -m teaser_check.experiment "{LAB}" --rubric skills/experiments/t3n.md --repeats 3 --model gemini-3.1-flash-lite --store "{STORE}" --out-dir "{OUT}"

### Step 7f. Rubric t4bn (t4b without the finding hypothesis_unlabelled), model gemini-3.1-flash-lite
One change against t4b, same model. Look for `rubrics ['t4bn']`.

In [ ]:
%cd /content/claim-check
!git pull -q origin lab-step3
import os
assert os.path.exists("skills/experiments/t4bn.md"), "t4bn.md is missing: git pull did not bring it"
!python -m teaser_check.experiment "{LAB}" --rubric skills/experiments/t4bn.md --repeats 3 --model gemini-3.1-flash-lite --store "{STORE}" --out-dir "{OUT}"

### Step 7g. The same rubric on a stronger model (gemini-3.5-flash-lite)
Set `BEST` to the rubric that did best in 7e/7f (default t4bn). One change: only the model. The model may answer 503: the run waits and tries again.

In [ ]:
%cd /content/claim-check
!git pull -q origin lab-step3
BEST = "skills/experiments/t4bn.md"   # change to t3n.md if it did better
import os
assert os.path.exists(BEST), BEST + " is missing"
!python -m teaser_check.experiment "{LAB}" --rubric "{BEST}" --model gemini-3.5-flash-lite --repeats 3 --store "{STORE}" --out-dir "{OUT}"

### Step 8. Save the results
Everything in Colab is deleted when the session ends. This downloads the tables and all run folders (zip). Do it before you close the tab, and send the zip to me: I score everything against the standard v2 again.

In [ ]:
%cd /content/claim-check
from google.colab import files
!zip -qr /content/teaser_experiments.zip "{OUT}/experiments_v2.csv" "{OUT}/experiment_runs"
files.download("/content/teaser_experiments.zip")